<a href="https://colab.research.google.com/github/EliVil2/RUTAS-IO/blob/codigo-distancia-total-146%2C11/Intento4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
pip install ortools folium openrouteservice geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [6]:
import folium
import openrouteservice
from openrouteservice import convert
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
from geopy.distance import geodesic
import numpy as np

# === CONFIGURACIÓN ===
ORS_API_KEY = '5b3ce3597851110001cf6248b75b48749e4e41a38cdaea0746c5ae37'
client = openrouteservice.Client(key=ORS_API_KEY)

# === DATOS DE ENTRADA ===
locations = [
    (10.983724, -74.789957),  # origen Universidad sede centro
    (10.9700532, -74.80195069999999), (10.9525577, -74.80394919999999), (10.975318, -74.808005),
    (10.9066255, -74.7856385), (10.9724651, -74.80604559999999), (11.017676, -74.809716),
    (10.944379, -74.803011), (10.9124556, -74.7826908), (10.9606144, -74.8028657),
    (10.9230761, -74.8008353), (10.9628562, -74.83497729999999), (10.940902, -74.771705),
    (10.9942585, -74.8122456), (10.963991, -74.81723699999999), (11.027218, -74.869495),
    (10.9615811, -74.83029429999999), (10.979663, -74.80364999999999), (10.9992743, -74.7924869),
    (10.9625489, -74.83082399999999), (10.985877, -74.83556), (11.024694, -74.86955499999999),
    (10.932394, -74.766357), (11.023623, -74.80664), (10.9030214, -74.7919232), (11.02425,-74.86800),
    (10.9558219, -74.8218484), (10.9599135, -74.8071882), (10.894661, -74.885339),
    (10.9590029, -74.796548), (10.7465427, -74.7565432), (11.0218788, -74.8705983),
]

demands = [0, 107, 112, 95, 84, 72, 107, 52, 110, 49, 67, 75, 107, 111, 107, 80, 87, 94, 78, 66, 30,
           70, 69, 94, 70, 114, 64, 46, 76, 33, 60, 69]

vehicle_capacity = 500
min_vehicles = 5

# === FUNCIONES AUXILIARES ===
def create_distance_matrix(locations):
    size = len(locations)
    matrix = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            if i == j:
                matrix[i][j] = 0
            else:
                matrix[i][j] = geodesic(locations[i], locations[j]).km
    return matrix

distance_matrix = create_distance_matrix(locations)

# === MODELO OR-TOOLS ===
def create_data_model():
    data = {}
    data['distance_matrix'] = (distance_matrix * 1000).astype(int)  # convertir a metros y a int
    data['demands'] = demands
    data['vehicle_capacities'] = [vehicle_capacity] * max(min_vehicles, int(np.ceil(sum(demands)/vehicle_capacity)))
    data['num_vehicles'] = len(data['vehicle_capacities'])
    data['depot'] = 0
    return data

data = create_data_model()

manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Capacidad vehículos
def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,  # no slack
    data['vehicle_capacities'],  # capacidad máxima
    True,  # start cumul to zero
    'Capacity'
)

# Evitar regreso al depot (rutas abiertas): para ello, permite que ruta termine en cualquier nodo, sin forzar regreso
# OR-Tools no soporta directamente rutas abiertas, pero se puede simular agregando un "penalty" alto a regresar al depot

penalty = 1000000  # penalización alta para saltar nodos (no forzar regreso)
for node in range(1, len(locations)):
    routing.AddDisjunction([manager.NodeToIndex(node)], penalty)

# Parámetros de búsqueda
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)  # solución inicial rápida
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)  # mejora solución
search_parameters.time_limit.seconds = 60  # tiempo máximo de búsqueda

# Resolver
solution = routing.SolveWithParameters(search_parameters)

if not solution:
    print("No se encontró solución.")
    exit()

# === EXTRAER RUTAS ===
def get_routes(solution, routing, manager):
    routes = []
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route.append(node_index)
            index = solution.Value(routing.NextVar(index))
        # No agregamos el depot al final (rutas abiertas)
        routes.append(route)
    return routes

routes = get_routes(solution, routing, manager)

# === FUNCIONES PARA RUTAS REALES EN MAPA ===
def get_real_route(coords):
    try:
        route = client.directions(coords, profile='driving-car', format='geojson')
        distance = route['features'][0]['properties']['segments'][0]['distance']
        geometry = route['features'][0]['geometry']
        return distance, geometry
    except Exception as e:
        print("Error con ruta:", coords, e)
        return float('inf'), None

def generar_enlace_google_maps(locations, route):
    base_url = "https://www.google.com/maps/dir/"
    waypoints = [f"{locations[i][0]},{locations[i][1]}" for i in route]
    return base_url + "/".join(waypoints)

# === CREAR MAPA ===
m = folium.Map(location=locations[0], zoom_start=12)
colors = ['blue', 'green', 'red', 'purple', 'orange', 'darkred', 'cadetblue', 'darkgreen', 'pink', 'lightblue']

total_distance = 0
all_routes = []

for i, route in enumerate(routes):
    if len(route) == 0:
        continue
    # calcular peso total
    total_weight = sum(demands[node] for node in route)

    # obtener distancias y geometrías reales
    distance = 0
    for j in range(len(route) - 1):
        segment = [tuple(reversed(locations[route[j]])), tuple(reversed(locations[route[j + 1]]))]
        dist, geom = get_real_route(segment)
        distance += dist
        if geom:
            folium.GeoJson(geom, name=f'Vehículo {i + 1}', style_function=lambda x, c=colors[i % len(colors)]: {'color': c, 'weight': 4}).add_to(m)

    total_distance += distance
    enlace = generar_enlace_google_maps(locations, route)
    all_routes.append((i + 1, route, total_weight, distance / 1000, enlace))

# === MOSTRAR RESULTADOS ===
for veh_id, route, weight, dist_km, enlace in all_routes:
    print(f"Vehículo {veh_id}: Ruta: {route} | Peso total: {weight} kg | Distancia: {dist_km:.2f} km")
    print(f"Google Maps: {enlace}\n")

m.save("rutas_vehiculos_optimizado.html")
print("Mapa guardado como rutas_vehiculos_optimizado.html")


/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 3rd time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 4th time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 5th time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rate limit exceeded. Retrying for the 6th time.
  warnings.warn('Rate limit exceeded. Retrying for the {0}{1} time.'.format(retry_counter + 1,
/usr/local/lib/python3.11/dist-packages/openrouteservice/client.py:211: UserWarning: Rat

Error con ruta: [(-74.868, 11.02425), (-74.869495, 11.027218)] 
Error con ruta: [(-74.869495, 11.027218), (-74.86955499999999, 11.024694)] 403 ({'error': 'Quota exceeded'})
Error con ruta: [(-74.86955499999999, 11.024694), (-74.8705983, 11.0218788)] 403 ({'error': 'Quota exceeded'})
Error con ruta: [(-74.8705983, 11.0218788), (-74.83556, 10.985877)] 403 ({'error': 'Quota exceeded'})
Error con ruta: [(-74.83556, 10.985877), (-74.83497729999999, 10.9628562)] 403 ({'error': 'Quota exceeded'})
Error con ruta: [(-74.83497729999999, 10.9628562), (-74.8071882, 10.9599135)] 403 ({'error': 'Quota exceeded'})
Vehículo 1: Ruta: [0, 29, 10, 8, 4, 24, 30, 28] | Peso total: 500 kg | Distancia: 73.23 km
Google Maps: https://www.google.com/maps/dir/10.983724,-74.789957/10.9590029,-74.796548/10.9230761,-74.8008353/10.9124556,-74.7826908/10.9066255,-74.7856385/10.9030214,-74.7919232/10.7465427,-74.7565432/10.894661,-74.885339

Vehículo 2: Ruta: [0, 1, 9, 2, 7, 22, 12] | Peso total: 496 kg | Distancia: 1